In [6]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F
import random 
import pickle 

In [7]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")  # works only with NVIDIA GPUs (not on Mac)
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [8]:
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=1):
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), 1))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                self.X[ii,jj] = \
                ord(data[ii+jj])-65
      
            self.y[ii] = \
                ord(data[ii+jj+1])-65

        self.X = tnsr(self.X).long()
        self.y = tnsr(self.y).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [11]:
class RNNEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True, nonlinearity='tanh', num_layers=2)
        self.linear = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x, h=None):
        embedded = self.embedding(x)
        out, h = self.rnn(embedded, h)
        out = self.linear(out[:,-1,:])

        return out, h  

In [14]:
reps = 10
short_term_memories = [2,4,6,8,10]
### initial training ###
total_samples = 50000
working_memory = 1
# short_term_memory = 10
hidden_size = 75
vocab_size = 7
embedding_dim = 5
lr = 3e-3
res = {}


for short_term_memory in short_term_memories:
    print('Doing BPTT ', short_term_memory)
    res[short_term_memory] = []

    for rep in range(reps):
        model = RNNEncoder(vocab_size, embedding_dim, hidden_size)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-8)
        criterion = torch.nn.CrossEntropyLoss()

        data = get_sequence(total_samples, 2, 3, train_percent=1.0)

        data_set = Dataset_converter(data, working_memory, short_term_memory)
        train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

        total = 0
        test_acc = []
        correct = np.zeros(1000,dtype=float)
        for X, y in train_loader:
            optimizer.zero_grad()

            if total == 0:
                y_pred, h = model(X)
            else:
                y_pred, h = model(X, hidden)

            loss = criterion(y_pred, y[0])     
            loss.backward()
            optimizer.step()

            # print(total)
            with torch.no_grad():
                hidden = h.detach()
                total += 1

                if y[0] == y_pred[0].argmax():
                        correct[total%1000] = 1
                else:
                    correct[total%1000] = 0


                test_acc.append(
                     np.sum(correct)/total if total<1000 else np.sum(correct)/1000
                )
                
                
                if total%10000 == 0:
                    print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')
        res[short_term_memory].append(test_acc)

with open('../pickle_files/naive_rnn.pickle', 'wb') as f:
    pickle.dump(res, f)

Doing BPTT  2
Iter : 10001, loss: 0.2311, accuracy: 0.6650
Iter : 20001, loss: 0.2110, accuracy: 0.6580
Iter : 30001, loss: 1.9560, accuracy: 0.6640
Iter : 40001, loss: 1.0190, accuracy: 0.6750
Iter : 10001, loss: 0.4309, accuracy: 0.6630
Iter : 20001, loss: 0.3976, accuracy: 0.6690
Iter : 30001, loss: 1.5573, accuracy: 0.6600
Iter : 40001, loss: 0.9475, accuracy: 0.6700
Iter : 10001, loss: 0.1958, accuracy: 0.6730
Iter : 20001, loss: 0.5327, accuracy: 0.6620
Iter : 30001, loss: 2.2051, accuracy: 0.6590
Iter : 40001, loss: 1.3408, accuracy: 0.6740
Iter : 10001, loss: 0.2008, accuracy: 0.6580
Iter : 20001, loss: 0.4071, accuracy: 0.6710
Iter : 30001, loss: 1.6934, accuracy: 0.6660
Iter : 40001, loss: 1.3274, accuracy: 0.6740
Iter : 10001, loss: 0.3209, accuracy: 0.6660
Iter : 20001, loss: 0.5385, accuracy: 0.6660
Iter : 30001, loss: 2.6648, accuracy: 0.6580
Iter : 40001, loss: 0.8738, accuracy: 0.6690
Iter : 10001, loss: 0.2301, accuracy: 0.6540
Iter : 20001, loss: 0.4430, accuracy: 0.6